In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, KFold
from sklearn.base import clone
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.feature_selection import RFE
from sklearn.utils import resample  # For simple oversampling (no imblearn needed)
import seaborn as sns
from pathlib import Path
import pickle
import xgboost as xgb
import matplotlib.pyplot as plt
from scipy.stats import linregress
from matplotlib import pyplot as plt

In [ ]:
# Surface tension integration
SIGMA = 0.072  # N/m (air-water, 20°C)
G = 9.81  # m/s²
RHO_L = 1000  # kg/m³
MU_L = 0.001  # Pa·s
U_L = 0.05  # m/s
U_B = 0.25  # m/s
Q_FLUX = 1e5  # W/m²
DELTA_T = 10  # K

In [ ]:
def predict_from_features(feat_dict, target='v'):
    with open("models/scaler_v.pkl", "rb") as f: 
        scaler_v = pickle.load(f)
    with open("models/scaler_as.pkl", "rb") as f: 
        scaler_as = pickle.load(f)
    with open("models/poly.pkl", "rb") as f: 
        poly = pickle.load(f)
    with open("models/top_feats_v.pkl", "rb") as f: 
        top_feats_v = pickle.load(f)
    with open("models/top_feats_as.pkl", "rb") as f: 
        top_feats_as = pickle.load(f)
    with open("models/ensemble_v_xgb.pkl", "rb") as f: 
        ensemble_v = pickle.load(f)
    with open("models/ensemble_as_xgb.pkl", "rb") as f: 
        ensemble_as = pickle.load(f)
    if target == 'v':
        feats = top_feats_v; model = ensemble_v; scaler_target = scaler_v
    else:
        feats = top_feats_as; model = ensemble_as; scaler_target = scaler_as
    
    x_df = pd.DataFrame([feat_dict])[feats].fillna(0)
    x_sc = scaler_target.transform(x_df.values)
    x_poly = poly.transform(x_sc)
    pred_log = model.predict(x_poly)[0]
    ml_pred = np.expm1(pred_log)
    
    d_eq = feat_dict.get('d_eq', 0)
    ar = feat_dict.get('AR', 1.0)
    phys_weight = 0.1 / ar
    if target == 'v':
        phys_pred = (np.pi / 6) * d_eq ** 3
    else:
        phys_pred = 4 * np.pi * (d_eq / 2) ** 2
    
    final_pred = (1 - phys_weight) * ml_pred + phys_weight * phys_pred
    return final_pred

In [ ]:
def prediction_results(data, count, view):
    test_df = data.copy()
    data_columns = test_df.columns
    FEATURES = list(data_columns[:-2])
    FEATURES_ext = FEATURES + ['vol_proxy', 'sa_proxy']
    test_df['vol_proxy'] = test_df['d_eq'] ** 3
    test_df['sa_proxy'] = np.pi * test_df['d_eq'] ** 2

    accuracy_data = []
    for i in range(len(test_df)):
        sample_dict = test_df[FEATURES_ext].iloc[i].to_dict()
        actual_v = test_df['Volume'].iloc[i]
        pred_v = predict_from_features(sample_dict, 'v')
        mape_v = abs(actual_v - pred_v) / actual_v * 100 if actual_v != 0 else 0
        actual_as = test_df['Surface_Area'].iloc[i]
        pred_as = predict_from_features(sample_dict, 'as')
        mape_as = abs(actual_as - pred_as) / actual_as * 100 if actual_as != 0 else 0
        accuracy_data.append({ 
            "Actual Volume": actual_v, "Prediction Volume": pred_v, "MAPE Volume": mape_v, 
            "Actual Surface Area": actual_as, "Predicted Surface Area": pred_as, "MAPE Surface Area": mape_as
        })

    Prediction_Results = pd.DataFrame(accuracy_data)
    print(Prediction_Results.describe())
    Prediction_Results.to_csv('Analysis bubbles/results/test_results_'+ view + str(count) + '.csv', index=False)

In [ ]:
for i in range(1, 11):
    print("Prediction started ....")
    data1 = pd.read_csv("Analysis bubbles/frame_black_Data"+ str(i) + ".csv", index_col=0)
    data2 = pd.read_csv("Analysis bubbles/frame_color_Data"+ str(i) + ".csv", index_col=0)
    prediction_results(data1, i, "black")
    prediction_results(data2, i, "color")
    print("Prediction done!")


In [ ]:
def physical_parameters_prediction(df, pixel_mm):
    rho_L     = 998.0      # kg/m³
    mu        = 1.002e-3    # Pa·s
    sigma     = 0.0728      # N/m
    g         = 9.81        # m/s²
    delta_rho = 997.0       # kg/m³ (air-water)
    k_L       = 0.598       # W/m·K
    Pr        = 7.0
    
    # ------------------- 3. Geometry -------------------
    df['d_eq_act_mm']  = (6 * df['Actual Volume']   / np.pi)**(1/3)
    df['d_eq_pred_mm'] = (6 * df['Prediction Volume']/ np.pi)**(1/3)

    # Sphericity ψ and deformation δ
    df['psi_act']  = (np.pi**(1/3) * (6*df['Actual Volume'])**(2/3))   / df['Actual Surface Area']
    df['psi_pred'] = (np.pi**(1/3) * (6*df['Prediction Volume'])**(2/3)) / df['Predicted Surface Area']
    df['delta_act']  = 1 - df['psi_act']
    df['delta_pred'] = 1 - df['psi_pred']
    act_deform = df['delta_act'].mean()
    pred_deform = df['delta_pred'].mean()

    # Sauter mean diameter d₃₂ (mm)
    d32_act_mm  = 6 * df['Actual Volume'].sum()    / df['Actual Surface Area'].sum()
    d32_pred_mm = 6 * df['Prediction Volume'].sum() / df['Predicted Surface Area'].sum()

    # ------------------- 4. Rise velocity & drag – Tomiyama (clean system) -------------------
    def tomiyama_clean(d_mm):
        d  = d_mm / 1000.0                     # m
        Eo = g * delta_rho * d**2 / sigma
        # Clean-system ellipsoidal regime (Tomiyama et al., 1998)
        Cd = (8/3) * Eo / (Eo + 4.0)
        Ub = np.sqrt( (4 * g * delta_rho * d) / (3 * rho_L * Cd) )
        return Ub, Cd, Eo

    Ub_act,  Cd_act,  Eo_act  = tomiyama_clean(d32_act_mm)
    Ub_pred, Cd_pred, Eo_pred = tomiyama_clean(d32_pred_mm)

    # Constants for your setup
    subchannel_width = 360 * pixel_mm  # mm
    subchannel_height = 1024 * pixel_mm # mm (Field of view height)
    # Correct V_sub: Ensure the width calculation reflects your actual subchannel geometry
    V_sub = (subchannel_width**2) * subchannel_height 

    # 1. Deduce Void Fraction (alpha)
    alpha_pred = df['Prediction Volume'].sum() / V_sub
    alpha_act  = df['Actual Volume'].sum() / V_sub

    # 2. Calculate IAC (a_i) in SI Units (m^-1)
    # Multiply by 1000 to convert from mm^-1 to m^-1
    ai_act  = (6 * alpha_act / d32_act_mm) * 10000
    ai_pred = (6 * alpha_pred / d32_pred_mm) * 10000

    # 3. Rename output for clarity
    # actual_values now contains the physical IAC (ai_act) in m^-1
    actual_values = [d32_act_mm, act_deform, Ub_act, Cd_act, h_act, ai_act]
    predicted_values = [d32_pred_mm, pred_deform, Ub_pred, Cd_pred, h_pred, ai_pred]
    
    return actual_values, predicted_values

In [ ]:
deformation_black = []
deformation_color = []
deformation_act = []
d_32_black = []
d_32_color = []
d_32_act = []
rise_velocity_black = []
rise_velocity_color = []
rise_velocity_act = []
int_drag_coef_black = []
int_drag_coef_color = []
int_drag_coef_act = []
heat_trans_coef_black = []
heat_trans_coef_color = []
heat_trans_coef_act = []
int_area_conc_black = []
int_area_conc_color = []
int_area_conc_act = []
for i in range(1, 11):
    print("calculations started ....")
    data1 = pd.read_csv("Analysis bubbles/results/test_results_black"+ str(i) + ".csv")
    data2 = pd.read_csv("Analysis bubbles/results/test_results_color"+ str(i) + ".csv")
    actual_black, predicted_black = physical_parameters_prediction(data1, 0.074)
    actual_color, predicted_color = physical_parameters_prediction(data2, 0.074)
    deformation_black.append(predicted_black[1])
    deformation_color.append(predicted_color[1])
    deformation_act.append(actual_black[1])
    d_32_black.append(predicted_black[0])
    d_32_color.append(predicted_color[0])
    d_32_act.append(actual_black[0])
    rise_velocity_black.append(predicted_black[2])
    rise_velocity_color.append(predicted_color[2])
    rise_velocity_act.append(actual_black[2])
    int_drag_coef_black.append(predicted_black[3])
    int_drag_coef_color.append(predicted_color[3])
    int_drag_coef_act.append(actual_black[3])
    heat_trans_coef_black.append(predicted_black[4])
    heat_trans_coef_color.append(predicted_color[4])
    heat_trans_coef_act.append(actual_black[4])
    int_area_conc_black.append(predicted_black[5])
    int_area_conc_color.append(predicted_color[5])
    int_area_conc_act.append(actual_black[5])
    print("calculations done!")

In [ ]:
lit_vals = [1.8, 0.08, 0.25, 1.0, 6.0, 2000]


In [ ]:
sauter_lit = [lit_vals[0]] * len(d_32_black)
# Update the global default parameters
plt.rcParams.update({'font.size': 28})
plt.figure(figsize=(12, 8))
plt.plot(d_32_black, label="pred view 1", linewidth=3, marker='o', markersize=10)
plt.plot(d_32_color, label="pred view 2", linewidth=3, marker='s', markersize=10)
plt.plot(d_32_act, label="actual", linewidth=3, marker='^', markersize=10)
plt.xlabel("Image Frames", fontsize=27)
plt.ylabel("Sauter Mean Diameter (mm)", fontsize=27)
plt.ylim(4.25, 5.5)  
plt.legend()
ax = plt.gca()
plt.setp(ax.spines.values(), color='black', linewidth=2)
plt.grid(True, linestyle='--')
plt.show()

In [ ]:
# Update the global default parameters
plt.rcParams.update({'font.size': 28})
plt.figure(figsize=(12, 8))
plt.plot(deformation_black, label="pred view 1", linewidth=3, marker='o', markersize=10)
plt.plot(deformation_color, label="pred view 2", linewidth=3, marker='s', markersize=10)
plt.plot(deformation_act, label="actual", linewidth=3, marker='^', markersize=10)
plt.xlabel("Image Frames", fontsize=27)
plt.ylabel("Bubble Deformation", fontsize=27)
plt.ylim(-0.025, 0.11)  
plt.legend()
ax = plt.gca()
plt.setp(ax.spines.values(), color='black', linewidth=2)
plt.grid(True, linestyle='--')
plt.show()

In [ ]:
# Update the global default parameters
plt.rcParams.update({'font.size': 28})
plt.figure(figsize=(12, 8))
plt.plot(rise_velocity_black, label="pred view 1", linewidth=3, marker='o', markersize=10)
plt.plot(rise_velocity_color, label="pred view 2", linewidth=3, marker='s', markersize=10)
plt.plot(rise_velocity_act, label="actual", linewidth=3, marker='^', markersize=10)
plt.xlabel("Image Frames", fontsize=27)
plt.ylabel("Rise Velocity (ms⁻¹)", fontsize=27)
plt.ylim(0.23, 0.236)  
plt.legend()
ax = plt.gca()
plt.setp(ax.spines.values(), color='black', linewidth=2)
plt.grid(True, linestyle='--')
plt.show()

In [ ]:
# Update the global default parameters
plt.rcParams.update({'font.size': 28})
plt.figure(figsize=(12, 8))
plt.plot(int_area_conc_black, label="pred view 1", linewidth=3, marker='o', markersize=10)
plt.plot(int_area_conc_color, label="pred view 2", linewidth=3, marker='s', markersize=10)
plt.plot(int_area_conc_act, label="actual", linewidth=3, marker='^', markersize=10)
plt.xlabel("Image Frames", fontsize=27)
plt.ylabel("Interfacial Area Concentration (m⁻¹)", fontsize=27)
#plt.ylim(1200, 1500)  
plt.legend()
ax = plt.gca()
plt.setp(ax.spines.values(), color='black', linewidth=2)
plt.grid(True, linestyle='--')
plt.show()